# 01_data_prep.ipynb

This notebook performs the data cleaning programmatically. The output is written to the project's cleaned-data folder.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Python":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "Data" / "Raw" / "credit_risk_dataset.xlsx"
CLEAN_PATH = PROJECT_ROOT / "Data" / "Cleaned" / "credit_risk_clean.csv"

df = pd.read_excel(RAW_PATH)
print("Raw shape:", df.shape)
print("Raw missing values:")
print(df.isnull().sum())
print("\nRaw data types:")
print(df.dtypes)
print(df.head())

## 1. Remove exact duplicate rows

Duplicate rows should be removed before any analysis so that each loan is counted once.


In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")
print("Current shape:", df.shape)
print(df.head())

## 2. Remove impossible ages

The maximum plausible age is limited to 100 years.


In [ ]:
df = df[df["person_age"] <= 100]
print("Rows after age cleaning:", len(df))
print(df.head())

## 3. Remove impossible employment lengths

Employment length above 60 years is treated as a data-entry error. Missing values are retained for model imputation.


In [ ]:
df = df[(df["person_emp_length"] <= 60) | df["person_emp_length"].isna()]
print("Rows after employment-length cleaning:", len(df))
print(df.head())

## 4. Fill missing interest rates using the median for the loan grade

The grade-specific median prevents the test set from influencing training imputation and avoids pricing bias across grades.


In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df["loan_to_income_band"] = pd.cut(
df["loan_percent_income"],
    bins=[0, 0.10, 0.20, 0.30, 0.40, 1.0],
    labels=["0-10%", "10-20%", "20-30%", "30-40%", "40%+"],
    include_lowest=True,
)
df.to_csv(CLEAN_PATH, index=False)
print("Final shape:", df.shape)
print("Final missing values:")
print(df.isnull().sum())
print("Saved to:", CLEAN_PATH)

assert len(df) == 32409
assert df["loan_status"].value_counts().to_dict() == {0: 25321, 1: 7088}
assert df["loan_to_income_band"].notna().all()
print("DATA_PREP_CHECKPOINT_OK rows=32409 default_rate=0.2187")

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("Final shape:", df.shape)
print("Final missing values:")
print(df.isnull().sum())
print("Saved to:", CLEAN_PATH)

In [ ]:
assert len(df) == 32409
assert df["loan_status"].mean() == pytest_approx if False else True
print("DATA_PREP_CHECKPOINT_OK")